<a href="https://colab.research.google.com/github/Kayla-afk/Tugas-Kuliah-D4-Sains-Data-Terapan/blob/main/DM_UTS_3324600023_Kayla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## IMPORT LIBRARY

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn. neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

Library yang digunakan mencakup:

* pandas & numpy → manipulasi data
* MinMaxScaler → normalisasi
* KNN → klasifikasi
* accuracy_score → evaluasi error

## LOAD DATASET

In [ ]:
df = pd.read_csv('/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Dataset Titanic dimuat sebagai data utama yang akan digunakan untuk imputasi nilai Age dan training model klasifikasi

## AMBIL FITUR YANG DIBUTUHKAN

In [ ]:
data = df[['Sex', 'Age', 'Pclass', 'Fare', 'Survived']].copy()

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['Sex'] = le.fit_transform(data['Sex'])

Hanya fitur relevan yang diambil, yaitu:
* Sex, Pclass, Fare, Survived → predictor
* Age → target imputasi

## ENCODING DATA KATEGORIKAL

In [ ]:
le = LabelEncoder()
data['Sex'] = le.fit_transform(data['Sex'])

/tmp/ipykernel_622/255831826.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Sex'] = le.fit_transform(data['Sex'])


Karena model tidak bisa membaca string, maka "male/female" → diubah menjadi numerik (0,1)

## SPLIT DATA (TRAIN & MISSING)

In [ ]:
train_data = data[data['Age'].notnull()]
test_data = data[data['Age'].isnull()]

* train_data → data dengan Age tersedia
* test_data → data Age kosong (akan diimputasi)

## PISAHKAN FITUR DAN LABEL

In [ ]:
X_train = train_data[['Sex', 'Pclass', 'Fare', 'Survived']]
y_train = train_data['Age']

X_test = test_data[['Sex', 'Pclass', 'Fare', 'Survived']]

* X_train → fitur input
* y_train → target (Age)
* X_test → data yang akan diprediksi

## NORMALISASI (MIN-MAX 0-1)

In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Normalisasi ini dilakukan agar KNN berbasis jarak dan menghindari bias dari fitur dengan skala besar (Fare)

## KLASIFIKASI (3-NN)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=3)
knn.fit(X_train_scaled, y_train)

age_pred = knn.predict(X_test_scaled)

* Model menggunakan 3 tetangga terdekat
* Age diprediksi sebagai “kelas”

## IMPUTASI MISSING VALUE


In [ ]:
data.loc[data['Age'].isnull(), 'Age'] = age_pred

Nilai Age yang kosong:

→ diganti dengan hasil prediksi KNN

→ dataset menjadi lengkap (no missing value)

# LOAD TEST DATASET

In [ ]:
test_df = pd.read_csv('/titanic_test.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Dataset ini digunakan untuk mengevaluasi performa model klasifikasi

## DATA TRAIN BARU

In [ ]:
train_data2 = data[['Sex', 'Age', 'Pclass', 'Fare']]
train_label2 = data['Survived']

Setelah dilakukan imputasi, maka Age sudah lengkap dan bisa digunakan sebagai fitur

## MENYIAPKAN DATA TEST

In [ ]:
test_data2 = test_df[['Sex', 'Age', 'Pclass', 'Fare']].dropna()

le2 = LabelEncoder()
test_data2['Sex'] = le2.fit_transform(test_data2['Sex'])

* Data test dibersihkan dari missing value
* Encoding ulang dilakukan

## LOAD LABEL TEST

In [ ]:
test_data = test_df[['Sex', 'Age', 'Pclass', 'Fare']]

valid_index = test_data.dropna().index

test_data = test_data.loc[valid_index]
test_label = pd.read_csv('/titanic_testlabel.csv')
test_label = test_label.loc[valid_index]

Label disesuaikan dengan baris test_data agar sinkron

## NORMALISASI DATA BARU

In [ ]:
scaler2 = MinMaxScaler()

train_scaled2 = scaler2.fit_transform(train_data2)
test_scaled2 = scaler2.transform(test_data2)

Dilakukan normalisasi ulang agar konsisten antara trai dengan test.

## KLASIFIKASI SURVIVED

In [ ]:
knn2 = KNeighborsClassifier(n_neighbors=3)
knn2.fit(train_scaled2, train_label2)

class_result = knn2.predict(test_scaled2)

Model KNN disini digunakan untuk memprediksi Survived (0/1)

## HITUNG ERROR

In [ ]:
error = np.sum(class_result != test_label['Survived'])
error_ratio = (error / len(test_data2)) * 100

print("Jumlah Error:", error)
print("Error Ratio (%):", error_ratio)

Jumlah Error: 51
Error Ratio (%): 15.407854984894259


* Error = jumlah prediksi salah
* Error ratio = persentase kesalahan model

## KESIMPULAN
Metode klasifikasi menggunakan KNN (k=3) berhasil digunakan untuk mengimputasi nilai Age yang hilang dengan memanfaatkan pola dari fitur lain, kemudian dataset yang telah lengkap digunakan untuk membangun model klasifikasi Survived, di mana normalisasi terbukti penting untuk menjaga akurasi berbasis jarak, dan performa model dievaluasi melalui error ratio yang menunjukkan tingkat kesalahan prediksi terhadap data uji.